# 01 - Seismic Data Preparation & Exploratory Data Analysis (EDA)

## Overview
Continuous seismic recording stations monitor ground motion around the clock across three orthogonal spatial components:
- **Z (Vertical)**: Ground motion in the upward/downward direction.
- **N (North-South)**: Horizontal motion along the north-south axis.
- **E (East-West)**: Horizontal motion along the east-west axis.

The **STEAD (Stanford Earthquake Dataset)** is a global catalog containing millions of annotated three-component seismograms, including local earthquakes and diverse ambient seismic noise recordings (cultural noise, ocean microseisms, instrument glitches).

### Objectives in this Notebook
1. **Extract and Filter**: Isolate local earthquake traces ($M > 2.5, d \le 21\text{ km}$) and ambient noise traces using the project's [`Data_PreProcessing`](../Data_PreProcessing) module.
2. **Window & Align**: Standardize each trace to a fixed temporal window of $2000$ samples ($20\text{ s}$ at $100\text{ Hz}$), centering earthquake windows on the P-wave arrival.
3. **Standardize (Z-score)**: Apply sequence-wise, channel-wise Z-score normalization: $\tilde{X} = \frac{X - \mu}{\sigma + \epsilon}$.
4. **Exploratory Data Analysis (EDA)**:
   - Inspect multi-component waveforms for both earthquakes and noise.
   - Perform Frequency Domain Analysis via Fast Fourier Transform (FFT) and Power Spectral Density (PSD).
   - Verify normalization statistics across train, validation, and test splits.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal

# Ensure project root is in sys.path so we can import Config, Data_PreProcessing, etc.
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from Config.dataset_config import extractedSTEAD

print(f"Project root: {ROOT_DIR}")
print(f"Target dataset specification: {extractedSTEAD}")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Inspecting the Extraction Pipeline Logic

The project handles extraction via [`Data_PreProcessing/STEAD_Data_Extraction_V2.py`](../Data_PreProcessing/STEAD_Data_Extraction_V2.py).

### Key Filtering & Preprocessing Criteria:
- **Sampling Rate**: $100\text{ Hz}$ (Nyquist frequency = $50\text{ Hz}$).
- **Component Order**: `ZNE` (Vertical, North-South, East-West).
- **Local Earthquakes Filter**:
  - `trace_category == 'earthquake_local'`
  - `source_magnitude > 2.5`
  - `source_distance_km <= 21`
- **Noise Filter**:
  - `trace_category == 'noise'`
- **Windowing**:
  - Length: $N = 2000$ timesteps.
  - Earthquake: Windowed around `trace_p_arrival_sample` using `sbg.WindowAroundSample`.
  - Noise: Random $2000$-sample window using `sbg.RandomWindow`.
- **Normalization**:
  - Channel-wise Z-score normalization per sequence: $\tilde{X}_{c, t} = \frac{X_{c, t} - \mu_c}{\sigma_c + 10^{-6}}$.

In [ ]:
# Check data directory contents and array shapes
data_dir = ROOT_DIR / "data"
train_x_path = data_dir / "train" / "x.npy"
train_y_path = data_dir / "train" / "y.npy"
val_x_path   = data_dir / "validation" / "x.npy"
val_y_path   = data_dir / "validation" / "y.npy"
test_x_path  = data_dir / "test" / "x.npy"
test_y_path  = data_dir / "test" / "y.npy"

if not train_x_path.exists():
    print("Preprocessed .npy files not found. Running data extraction...")
    from Data_PreProcessing.STEAD_Data_Extraction_V2 import dataBuilder
    dataBuilder()
else:
    print("Preprocessed dataset already exists on disk.")

# Load preprocessed arrays into memory
X_train, y_train = np.load(train_x_path), np.load(train_y_path)
X_val, y_val     = np.load(val_x_path), np.load(val_y_path)
X_test, y_test   = np.load(test_x_path), np.load(test_y_path)

print("=== Dataset Dimensions & Partitions ===")
print(f"Training set:   X = {X_train.shape}, y = {y_train.shape}")
print(f"Validation set: X = {X_val.shape}, y = {y_val.shape}")
print(f"Test set:       X = {X_test.shape}, y = {y_test.shape}")
print(f"Channels:       {X_train.shape[1]} (Z, N, E)")
print(f"Timesteps:      {X_train.shape[2]} (20.0 seconds at 100 Hz)")

## 2. Dataset Partitioning & Class Balance

Let's confirm the class distribution across all splits to ensure perfectly stratified representation between class `0` (Ambient Noise) and class `1` (Earthquake).

In [ ]:
split_data = []
for split_name, y_arr in [("Train (80%)", y_train), ("Val (10%)", y_val), ("Test (10%)", y_test)]:
    n_noise = int(np.sum(y_arr == 0))
    n_eq    = int(np.sum(y_arr == 1))
    split_data.append({
        "Split": split_name,
        "Noise (0)": n_noise,
        "Earthquake (1)": n_eq,
        "Total": len(y_arr),
        "Earthquake Ratio": f"{n_eq / len(y_arr):.1%}"
    })

df_splits = pd.DataFrame(split_data)
display(df_splits)

# Plot split distribution
fig, ax = plt.subplots(figsize=(8, 4))
bar_w = 0.35
x_idx = np.arange(len(df_splits))
ax.bar(x_idx - bar_w/2, df_splits['Noise (0)'], bar_w, label='Noise (0)', color='#4A90E2')
ax.bar(x_idx + bar_w/2, df_splits['Earthquake (1)'], bar_w, label='Earthquake (1)', color='#E94E77')
ax.set_xticks(x_idx)
ax.set_xticklabels(df_splits['Split'])
ax.set_ylabel("Number of Samples")
ax.set_title("Stratified Class Balance across Splits")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Waveform Visualization (Earthquake vs. Noise)

Now let's examine individual seismograms:
- **Earthquake Traces**: Distinct arrival of the **P-wave** (compressional, prominent on vertical component Z), followed quickly by the higher-amplitude **S-wave** (shear, pronounced on horizontal components N and E), then decaying coda waves.
- **Noise Traces**: Stationary background vibrations, cultural microseisms, or random instrument noise without characteristic impulsive phase onsets.

In [ ]:
# Find indices of an earthquake and a noise sample
eq_idx = np.where(y_train == 1)[0][0]
noise_idx = np.where(y_train == 0)[0][0]

time_axis = np.linspace(0, 20.0, extractedSTEAD['n_length'])
channel_names = ['Z (Vertical)', 'N (North-South)', 'E (East-West)']
colors = ['#1f77b4', '#2ca02c', '#d62728']

fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)

# Plot Earthquake
for c in range(3):
    axes[c, 0].plot(time_axis, X_train[eq_idx, c, :], color=colors[c], lw=1.2)
    axes[c, 0].set_ylabel(channel_names[c])
    axes[c, 0].grid(True, alpha=0.3)
axes[0, 0].set_title(f"Local Earthquake Trace (Sample #{eq_idx})", fontsize=13, fontweight='bold', color='#E94E77')
axes[2, 0].set_xlabel("Time (seconds)")

# Plot Noise
for c in range(3):
    axes[c, 1].plot(time_axis, X_train[noise_idx, c, :], color=colors[c], lw=1.2)
    axes[c, 1].set_ylabel(channel_names[c])
    axes[c, 1].grid(True, alpha=0.3)
axes[0, 1].set_title(f"Ambient Noise Trace (Sample #{noise_idx})", fontsize=13, fontweight='bold', color='#4A90E2')
axes[2, 1].set_xlabel("Time (seconds)")

plt.tight_layout()
plt.show()

## 4. Frequency Domain Analysis: FFT & Power Spectral Density (PSD)

Earthquake signals contain distinct frequency bands (typically $1 - 25\text{ Hz}$ for local events), while seismic noise often has different spectral characteristics (e.g., microseisms around $0.1 - 1\text{ Hz}$, high-frequency anthropogenic noise above $20\text{ Hz}$).

Let's compute the average Power Spectral Density across earthquakes vs noise using Welch's method.

In [ ]:
fs = 100.0  # sampling rate (Hz)

# Compute Welch's PSD for vertical channel (Z)
eq_indices = np.where(y_train == 1)[0][:200]
noise_indices = np.where(y_train == 0)[0][:200]

def compute_mean_psd(indices, channel=0):
    psds = []
    for idx in indices:
        f, pxx = signal.welch(X_train[idx, channel, :], fs=fs, nperseg=256)
        psds.append(pxx)
    return f, np.mean(psds, axis=0), np.std(psds, axis=0)

freqs, eq_psd_mean, eq_psd_std = compute_mean_psd(eq_indices, channel=0)
_, noise_psd_mean, noise_psd_std = compute_mean_psd(noise_indices, channel=0)

plt.figure(figsize=(10, 5))
plt.semilogy(freqs, eq_psd_mean, label='Earthquakes (Vertical Component Z)', color='#E94E77', lw=2)
plt.fill_between(freqs, eq_psd_mean - 0.5*eq_psd_std, eq_psd_mean + 0.5*eq_psd_std, color='#E94E77', alpha=0.2)

plt.semilogy(freqs, noise_psd_mean, label='Noise (Vertical Component Z)', color='#4A90E2', lw=2)
plt.fill_between(freqs, noise_psd_mean - 0.5*noise_psd_std, noise_psd_mean + 0.5*noise_psd_std, color='#4A90E2', alpha=0.2)

plt.title("Average Power Spectral Density: Earthquake vs. Ambient Noise", fontsize=13, fontweight='bold')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power Spectral Density (V²/Hz)")
plt.xlim(0, 50)
plt.legend(fontsize=11)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Sequence Normalization Verification

Each trace was normalized channel-wise with Z-score standardization:
$$\mu_c = \frac{1}{L} \sum_{t=1}^L X_{c, t} \approx 0, \quad \sigma_c = \sqrt{\frac{1}{L} \sum_{t=1}^L (X_{c, t} - \mu_c)^2} \approx 1$$

Let's confirm the empirical mean and standard deviation across all samples and channels in the training set.

In [ ]:
means = np.mean(X_train, axis=2)  # shape: (N, 3)
stds  = np.std(X_train, axis=2)   # shape: (N, 3)

print("Mean across all traces (should be ~0.0):")
for c, name in enumerate(['Z', 'N', 'E']):
    print(f"  Channel {name}: mean = {np.mean(means[:, c]):.6f}, std of means = {np.std(means[:, c]):.6f}")

print("\nStd dev across all traces (should be ~1.0):")
for c, name in enumerate(['Z', 'N', 'E']):
    print(f"  Channel {name}: mean std = {np.mean(stds[:, c]):.6f}, std of stds = {np.std(stds[:, c]):.6f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(means.flatten(), bins=50, color='#2ca02c', alpha=0.7, edgecolor='black')
axes[0].set_title("Distribution of Sequence Means")
axes[0].set_xlabel("Mean Value")
axes[0].set_ylabel("Count")

axes[1].hist(stds.flatten(), bins=50, color='#1f77b4', alpha=0.7, edgecolor='black')
axes[1].set_title("Distribution of Sequence Standard Deviations")
axes[1].set_xlabel("Std Deviation Value")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

## Summary & Next Steps

We have verified that the seismic dataset is:
1. **Cleanly pre-processed** with 3 channels ($Z, N, E$) and 2000 timesteps ($20\text{ s}$) per sample.
2. **Properly stratified** with equal numbers of earthquake and noise traces in train, validation, and test sets.
3. **Standardized** with zero mean and unit variance per channel.

In the next notebook [`02_Pretraining_MAE_and_Clustering.ipynb`](02_Pretraining_MAE_and_Clustering.ipynb), we pretrain a **Masked Autoencoder (MAE)** on these unlabelled seismic waveforms and evaluate its representations via unsupervised clustering!